<a href="https://colab.research.google.com/github/beatrizzzzz/Estudo-com-o-PERMA-Profiler/blob/main/graficos_tcc_violinplot_radial_boxplot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Gráficos complementares — TCC PERMA

Este notebook contém três visualizações utilizadas no trabalho:

| # | Gráfico | Finalidade |
|---|---|---|
| 1 | **Violinplot** | Comparar a distribuição dos escores das dimensões PERMA entre os perfis de bem-estar |
| 2 | **Visualização radial** | Comparar o perfil médio das cinco dimensões centrais do PERMA por cluster |
| 3 | **Boxplot das dimensões externas** | Comparar Emoções Negativas, Saúde Física percebida, Solidão e Felicidade Geral entre os perfis |

> **Pré-requisito:** o arquivo `dados_tcc.csv` deve estar na mesma pasta deste notebook.


In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Configuração global dos gráficos
plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'font.size': 11,
    'axes.titlesize': 12,
    'axes.titleweight': 'bold',
    'axes.edgecolor': '#444444',
    'axes.linewidth': 0.8,
    'figure.dpi': 150,
    'savefig.dpi': 150,
    'savefig.bbox': 'tight',
})

# Paleta utilizada nos dois gráficos
AZUL = '#2E6E9E'
VERDE = '#3C9D6E'
LARANJA = '#E08A2B'

PALETA = {
    'Florescimento': VERDE,
    'Desenvolvimento': LARANJA,
}

# Carregar os dados
df = pd.read_csv('dados_tcc.csv')

print(f'Dados carregados: {len(df)} participantes')
print(df['Perfil'].value_counts())


## 1. Violinplot — distribuição dos perfis por dimensão PERMA

O gráfico apresenta a distribuição dos escores de cada dimensão, separada pelos perfis de bem-estar. Além da forma da distribuição, são exibidas a mediana e a amplitude interquartil.


In [ ]:
# Reorganizar os dados no formato longo para utilização pelo seaborn
dimensoes_violin = {
    'Escore_P': 'Em. Positivas (P)',
    'Escore_E': 'Engajamento (E)',
    'Escore_R': 'Relacionamentos (R)',
    'Escore_M': 'Sentido de Vida (M)',
    'Escore_A': 'Realização (A)',
    'Escore_EN': 'Em. Negativas (EN)',
}

df_long = df[list(dimensoes_violin.keys()) + ['Perfil']].melt(
    id_vars='Perfil',
    var_name='Dimensao_raw',
    value_name='Escore'
)

df_long['Dimensão'] = df_long['Dimensao_raw'].map(dimensoes_violin)
ordem_dimensoes = list(dimensoes_violin.values())

fig, ax = plt.subplots(figsize=(12, 5.5))

sns.violinplot(
    data=df_long,
    x='Dimensão',
    y='Escore',
    hue='Perfil',
    order=ordem_dimensoes,
    palette=PALETA,
    split=True,
    inner='box',
    linewidth=0.8,
    ax=ax,
)

# Linha de referência no ponto médio da escala
ax.axhline(
    3,
    color='#CC4444',
    linestyle='--',
    linewidth=1,
    label='Ponto médio (3,0)',
)

# Separação visual entre as dimensões positivas e as emoções negativas
ax.axvline(4.5, color='#BBBBBB', linestyle=':', linewidth=1.5)
ax.text(
    4.6,
    5.2,
    'EN (não\ndiferencia)',
    fontsize=8.5,
    color='#777777',
    style='italic',
)

ax.set_title('Distribuição dos escores por dimensão e perfil de bem-estar')
ax.set_ylabel('Escore (escala 1–5)')
ax.set_xlabel('')
ax.set_ylim(0.5, 5.6)
ax.legend(title='Perfil', loc='lower left')
ax.grid(axis='y', color='#E6E6E6', linewidth=0.7)
ax.set_axisbelow(True)

plt.tight_layout()
plt.savefig('violinplot_perfis.png')
plt.show()


## 2. Visualização radial — perfil médio por cluster no Modelo PERMA

O gráfico radial compara as médias das cinco dimensões do PERMA entre os perfis de florescimento e desenvolvimento, além de apresentar a média da amostra total e o ponto médio teórico da escala.


In [ ]:
# Dimensões e respectivas colunas do dataframe
rotulos_radiais = [
    'P\nEmoções\nPositivas',
    'E\nEngaja-\nmento',
    'R\nRelacio-\nnamentos',
    'M\nSentido\nde Vida',
    'A\nReali-\nzação',
]

colunas_perma = [
    'Escore_P',
    'Escore_E',
    'Escore_R',
    'Escore_M',
    'Escore_A',
]

# Calcular as médias diretamente a partir do dataframe
medias_por_perfil = df.groupby('Perfil')[colunas_perma].mean()

florescimento = medias_por_perfil.loc['Florescimento'].values
desenvolvimento = medias_por_perfil.loc['Desenvolvimento'].values
amostra_total = df[colunas_perma].mean().values

numero_dimensoes = len(rotulos_radiais)
angulos = np.linspace(0, 2 * np.pi, numero_dimensoes, endpoint=False)

# Fechar os polígonos repetindo o primeiro ponto
def fechar_poligono(valores):
    return np.append(valores, valores[0])

angulos_fechados = np.append(angulos, angulos[0])

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw={'polar': True})

# Posicionar a primeira dimensão no topo e organizar o sentido dos eixos
ax.set_theta_offset(np.pi / 2)
ax.set_theta_direction(-1)

# Grades e limites
ax.set_ylim(0, 5)
ax.set_yticks([1, 2, 3, 4, 5])
ax.set_yticklabels(['1', '2', '3', '4', '5'], fontsize=8, color='#666666')
ax.yaxis.set_tick_params(labelrotation=0)
ax.set_rlabel_position(90)

# Ponto médio teórico da escala
ax.plot(
    angulos_fechados,
    fechar_poligono(np.full(numero_dimensoes, 3)),
    color='#AAAAAA',
    linewidth=0.8,
    linestyle='--',
    zorder=2,
)

# Perfil de florescimento
ax.fill(
    angulos_fechados,
    fechar_poligono(florescimento),
    color=VERDE,
    alpha=0.25,
    zorder=3,
)
ax.plot(
    angulos_fechados,
    fechar_poligono(florescimento),
    color=VERDE,
    linewidth=2,
    zorder=4,
)

for angulo, raio, valor in zip(angulos, florescimento, florescimento):
    ax.text(
        angulo,
        raio + 0.18,
        f'{valor:.2f}',
        ha='center',
        va='center',
        fontsize=8.5,
        color=VERDE,
        fontweight='bold',
        zorder=5,
    )

# Perfil de desenvolvimento
ax.fill(
    angulos_fechados,
    fechar_poligono(desenvolvimento),
    color=LARANJA,
    alpha=0.25,
    zorder=3,
)
ax.plot(
    angulos_fechados,
    fechar_poligono(desenvolvimento),
    color=LARANJA,
    linewidth=2,
    zorder=4,
)

for angulo, raio, valor in zip(angulos, desenvolvimento, desenvolvimento):
    ax.text(
        angulo,
        raio + 0.18,
        f'{valor:.2f}',
        ha='center',
        va='center',
        fontsize=8.5,
        color=LARANJA,
        fontweight='bold',
        zorder=5,
    )

# Média da amostra total
ax.plot(
    angulos_fechados,
    fechar_poligono(amostra_total),
    color=AZUL,
    linewidth=1.5,
    linestyle='--',
    alpha=0.8,
    zorder=3,
)

# Eixos e rótulos
ax.set_xticks(angulos)
ax.set_xticklabels(rotulos_radiais, fontsize=9)
ax.spines['polar'].set_color('#CCCCCC')
ax.grid(color='#DDDDDD', linewidth=0.6)

# Título e legenda
n_florescimento = (df['Perfil'] == 'Florescimento').sum()
n_desenvolvimento = (df['Perfil'] == 'Desenvolvimento').sum()
n_total = len(df)

ax.set_title(
    'Perfil psicológico médio por cluster — Modelo PERMA\n'
    '(visualização radial das cinco dimensões do florescimento)',
    pad=20,
    fontsize=11,
    fontweight='bold',
)

elementos_legenda = [
    mpatches.Patch(
        facecolor=VERDE,
        edgecolor=VERDE,
        label=f'Florescimento (n={n_florescimento})',
    ),
    mpatches.Patch(
        facecolor=LARANJA,
        edgecolor=LARANJA,
        label=f'Desenvolvimento (n={n_desenvolvimento})',
    ),
    plt.Line2D(
        [0],
        [0],
        color=AZUL,
        linewidth=1.5,
        linestyle='--',
        label=f'Amostra total (n={n_total})',
    ),
    plt.Line2D(
        [0],
        [0],
        color='#AAAAAA',
        linewidth=0.8,
        linestyle='--',
        label='Ponto médio teórico (3,0)',
    ),
]

ax.legend(
    handles=elementos_legenda,
    loc='lower right',
    bbox_to_anchor=(1.35, -0.05),
    fontsize=8.5,
    framealpha=0.9,
)

plt.tight_layout()
plt.savefig('radar_perma.png')
plt.show()


## 3. Boxplot — dimensões externas ao agrupamento

A figura compara as quatro dimensões suplementares do PERMA-Profiler entre os perfis de desenvolvimento e florescimento. Essas variáveis são apresentadas como dimensões externas porque não foram utilizadas na formação dos agrupamentos.


In [ ]:
# Colunas das dimensões suplementares e títulos apresentados em cada painel
# EN = Emoções Negativas; SF = Saúde Física percebida;
# S = Solidão; FG = Felicidade Geral.
dimensoes_externas = {
    'Escore_EN': 'Emoções\nnegativas',
    'Escore_SF': 'Saúde física\npercebida',
    'Escore_S': 'Solidão',
    'Escore_FG': 'Felicidade\ngeral',
}

# Verificar se todas as variáveis necessárias estão presentes no arquivo
colunas_obrigatorias = ['Perfil', *dimensoes_externas.keys()]
colunas_ausentes = [coluna for coluna in colunas_obrigatorias if coluna not in df.columns]
if colunas_ausentes:
    raise KeyError(
        'As seguintes colunas necessárias para o boxplot não foram encontradas: '
        + ', '.join(colunas_ausentes)
    )

# Criar rótulos abreviados, mantendo a ordem Desenvolvimento → Florescimento
df_boxplot = df[colunas_obrigatorias].copy()
df_boxplot['Perfil_abrev'] = df_boxplot['Perfil'].map({
    'Desenvolvimento': 'Desenv.',
    'Florescimento': 'Floresc.',
})

# Remover registros cujo perfil não corresponda aos dois grupos analisados
df_boxplot = df_boxplot.dropna(subset=['Perfil_abrev'])

ordem_perfis = ['Desenv.', 'Floresc.']
cores_boxplot = ['#F2A66F', '#A6CB87']

# Criar quatro painéis alinhados e com a mesma escala vertical
fig, eixos = plt.subplots(
    nrows=1,
    ncols=4,
    figsize=(14.5, 5.2),
    sharey=True,
)

for indice, (coluna, titulo) in enumerate(dimensoes_externas.items()):
    ax = eixos[indice]

    sns.boxplot(
        data=df_boxplot,
        x='Perfil_abrev',
        y=coluna,
        order=ordem_perfis,
        palette=cores_boxplot,
        width=0.55,
        linewidth=1.2,
        boxprops={'edgecolor': '#4D4D4D'},
        whiskerprops={'color': '#4D4D4D', 'linewidth': 1.2},
        capprops={'color': '#4D4D4D', 'linewidth': 1.2},
        medianprops={'color': '#E47F32', 'linewidth': 1.4},
        flierprops={
            'marker': 'o',
            'markerfacecolor': 'white',
            'markeredgecolor': '#555555',
            'markeredgewidth': 1.2,
            'markersize': 5,
        },
        ax=ax,
    )

    # Linha horizontal correspondente ao ponto médio da escala de 1 a 5
    ax.axhline(
        y=3,
        color='#8C8C8C',
        linestyle='--',
        linewidth=0.9,
        zorder=0,
    )

    ax.set_title(titulo, fontsize=12, fontweight='normal', pad=8)
    ax.set_xlabel('')
    ax.set_ylim(0.8, 5.2)
    ax.set_yticks(np.arange(1.0, 5.1, 0.5))
    ax.grid(axis='y', color='#E9E9E9', linewidth=0.7)
    ax.set_axisbelow(True)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    if indice == 0:
        ax.set_ylabel('Escore (1 a 5)', fontsize=11)
    else:
        ax.set_ylabel('')

fig.suptitle(
    'Dimensões externas ao agrupamento, segundo o perfil de bem-estar',
    fontsize=14,
    fontweight='normal',
    y=1.02,
)

plt.tight_layout(rect=[0, 0, 1, 0.94], w_pad=1.2)
plt.savefig('boxplot_dimensoes_externas.png', dpi=300)
plt.show()
